# 02 · Preprocessing Pipeline
  
It covers every transformation applied to the raw car-sales dataset before model training:

| Step | Function | Purpose |
|------|----------|---------|
| 1 | `encode_categoricals()` | Convert string categories to numeric form |
| 2 | `split_data()` | Separate features / target, partition 80 / 20 |
| 3 | `scale_features()` | Standardise all numeric columns |

> **Why a dedicated preprocessing module?**  
> Keeping data-cleaning logic in one place makes it easy to reproduce the exact  
> same transformations at inference time and during cross-validation.


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## `encode_categoricals(df)`

### Design decisions

#### `transmission` → binary encoding (Manual = 0, Automatic = 1)
`transmission` has **exactly two** categories with no ambiguity:

| Raw value | Encoded |
|-----------|---------|
| `Manual` | `0` |
| `Automatic` | `1` |

Binary encoding preserves the single meaningful bit of information (is it automatic?) without inflating dimensionality.  
A runtime guard (`ValueError`) ensures the assumption of exactly-2 categories holds; if the upstream data ever changes, the pipeline fails loudly rather than silently producing wrong results.

#### `fuel` & `seller_type` → one-hot encoding (`pd.get_dummies`)
Both columns contain **more than two** unordered categories (e.g. fuel: Petrol, Diesel, CNG, LPG, Electric).  
Assigning integer codes (0, 1, 2 …) would imply a false ordinal relationship  
("Electric is greater than Diesel"), which would mislead any distance- or gradient-based learner.  
One-hot encoding creates one binary column per category, keeping each value  
independent.

#### `drop_first=True` — avoiding the dummy-variable trap
With *k* categories, one-hot encoding produces *k* columns, but only *k − 1* are  
linearly independent: the last is perfectly predictable from the others.  
This **multicollinearity** inflates variance in OLS and can destabilise gradient  
descent.  Dropping the first dummy eliminates redundancy at no cost to  
information content.


In [ ]:
def encode_categoricals(df: pd.DataFrame):
    df = df.copy()
    for col in ["transmission"]:
        categories = sorted(df[col].unique())
        if len(categories) != 2:
            raise ValueError(
                f"'{col}' expected exactly 2 categories, found {categories}"
            )
        mapping = {categories[0]: 0, categories[1]: 1}
        df[col] = df[col].map(mapping)
        print(f"Binary-encoded '{col}': {mapping}")

    df = pd.get_dummies(df, columns=["fuel", "seller_type"], drop_first=True)
    print(f"One-hot encoded: {['fuel', 'seller_type']}")
    print(f"Resulting columns: {list(df.columns)}")
    return df


### `split_data(df)`

#### Design decisions

#### Dropping `name`
`name` is a **free-text car model identifier** (e.g. `"Maruti Swift VXI"`,  
`"Honda City 1.5 EXi"`) with **hundreds of unique values**.

- It cannot be ordinally encoded — there is no meaningful numeric order between  
  car model names.  
- Naïve label-encoding would assign arbitrary integers that imply a ranking  
  that does not exist.  
- One-hot encoding would create hundreds of near-zero sparse columns, adding  
  noise and overfitting risk, especially for rare models seen only once.  
- Any information it contains (brand, segment, displacement) is already captured  
  by the other engineered features (`year`, `km_driven`, `fuel`, `transmission`).


#### 80 / 20 train-test split with `random_state=42`
- **80 % train** gives the model enough examples to learn generalizable patterns.  
- **20 % test** provides a held-out set large enough (≈ 1 500 rows on a  
  ~7 500-row dataset) to produce statistically stable evaluation metrics.  
- **`random_state=42`** makes the split deterministic: any team member or CI run  
  reproduces identical partitions, so benchmark numbers are comparable across  
  experiments.


In [ ]:
def split_data(df: pd.DataFrame):
    if "name" in df.columns:
        print("Dropping 'name': free-text car model identifier — "
              "hundreds of unique values, no ordinal meaning, "
              "not suitable as a predictive feature.")

    cols_to_drop = ["selling_price"] + [c for c in ["name"] if c in df.columns]
    X = df.drop(columns=cols_to_drop)
    y = df["selling_price"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print(f"\nTrain/test split: X_train={X_train.shape}, X_test={X_test.shape}")
    return X_train, X_test, y_train, y_test


## `scale_features(X_train, X_test)`

### Design decisions

#### Why `StandardScaler` and not `MinMaxScaler`?
`StandardScaler` transforms each feature to **zero mean, unit variance**:

$$x' = \frac{x - \mu}{\sigma}$$

`MinMaxScaler` compresses values to `[0, 1]` using the observed minimum and  
maximum, making it **sensitive to outliers**: a single extreme `km_driven` value  
would compress all other values into a tiny range.  
Car-sales data is notoriously skewed (luxury outliers, salvage cars), so  
StandardScaler's statistics (mean, std) are more robust.

Additionally, models like **SVR and Ridge regression** rely on Euclidean distance  
or regularisation penalties that assume features are on comparable scales.  
StandardScaler is the canonical choice for these algorithms.

#### Why `fit` only on the training set?
If we called `fit_transform` on the entire dataset, the scaler's mean and  
standard deviation would incorporate information from the test rows.  
The test set would then **leak future information** into the learned parameters —  
a subtle form of data leakage that causes over-optimistic evaluation metrics.  
Fitting on training data only ensures the test set is treated as genuinely  
unseen.

#### Why `transform` both sets with the **same fitted scaler**?
At inference time the model will receive data that must be scaled with the  
**same statistics** used during training.  
Re-fitting a separate scaler on the test set would produce different μ / σ  
values, effectively putting test features on a different scale to what the model  
was trained on — guaranteeing degraded predictions.  
Returning the fitted `scaler` object also makes it straightforward to pickle and  
reuse it in a production serving pipeline.


In [ ]:
def scale_features(X_train, X_test):
    scaler = StandardScaler()
    cols = X_train.columns.tolist()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=cols, index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test), columns=cols, index=X_test.index
    )

    print(f"\nStandardScaler fitted on training set ({len(cols)} features).")
    print(f"  mean (first 4): {np.round(scaler.mean_[:4], 4)}")
    print(f"  std  (first 4): {np.round(scaler.scale_[:4], 4)}")
    return X_train_scaled, X_test_scaled, scaler
